In [ ]:
from google.colab import drive
from pathlib import Path
import os
import subprocess

drive.mount('/content/drive')
os.environ['AQI_DATA_DIR'] = '/content/drive/MyDrive/AQI_Project_Data'
REPO_DIR = Path('/content/AQI-Air-_pollution_prediction')
REPO_URL = 'https://github.com/GOKUL5287/AQI-Air-_pollution_prediction.git'

if (REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
elif REPO_DIR.exists():
    raise FileExistsError(f'{REPO_DIR} exists but is not a Git repository')
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print('Repository:', REPO_DIR)
print('Shared data:', os.environ['AQI_DATA_DIR'])

# D3 AQI baselines

Requires D2 at `AQI_DATA_DIR/processed/features_v1.parquet`. Each horizon is trained only on `split == 'train'` rows with an available target, and scored on target-valid `split == 'test'` rows.

In [ ]:
from src.data_loader import DATA_DIR, load_features
from src.baselines import run_baselines
from src.evaluate import save_outputs

features = load_features()
print(f'D2 shape: {features.shape}')
print(features['split'].value_counts(dropna=False).sort_index())

In [ ]:
predictions, metrics = run_baselines(features)
predictions_path, metrics_path = save_outputs(
    predictions, metrics, DATA_DIR / 'processed'
)
print('Predictions:', predictions_path)
print('Metrics:', metrics_path)
display(metrics.sort_values(['horizon_h', 'subset', 'model']).reset_index(drop=True))

In [ ]:
comparison = []
test_all = metrics.loc[metrics['subset'] == 'test_all']
for horizon in (24, 48):
    scores = test_all.loc[test_all['horizon_h'] == horizon].set_index('model')['MAE']
    status = 'beats' if scores['hist_gradient_boosting'] < scores['persistence'] else 'does not beat'
    comparison.append(
        f"{horizon}h {status} persistence "
        f"(MAE {scores['hist_gradient_boosting']:.2f} vs {scores['persistence']:.2f})"
    )
print('Gradient boosting ' + '; '.join(comparison) + ' on test_all.')